# 01 — Data, preprocessing, and Frangi

Retinal vessels are a **screening biomarker**, not just a segmentation toy. Calibre, density and tortuosity change with diabetic retinopathy and hypertension. DRIVE (Staal *et al.*, 2004) is the 40-image fundus benchmark this repo is built around.

This notebook uses **synthetic DRIVE-layout images** so it runs without a Grand Challenge download. Official photographs go in `data/drive/` — see `data/README.md`. We do **not** redistribute DRIVE.

Charts are **Plotly only**. Interactive overlays: Napari in notebook 04.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[0] / "src") if Path.cwd().name == "notebooks" else "src")

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from drive_seg.classical import segment_frangi, vesselness
from drive_seg.data import iter_samples, load_sample, resolve_dataset_root
from drive_seg.metrics import segmentation_metrics
from drive_seg.preprocess import green_channel, prepare_gray
from drive_seg.synthetic import generate_synthetic_dataset
from drive_seg.paths import SYNTHETIC_DIR

if not (SYNTHETIC_DIR / "training" / "images").is_dir():
    generate_synthetic_dataset()
root = resolve_dataset_root()
print("dataset root:", root)
samples = iter_samples(root, split="test")
print(f"{len(samples)} test images")
sample = samples[0]
image, vessel, fov = load_sample(sample)
print(sample.sample_id, image.shape, "vessel frac", float(vessel.mean()))

## Green channel + CLAHE

DRIVE photographs are RGB, but the **green channel** carries most of the vessel contrast (red is saturated, blue is noisy). Contrast-limited adaptive histogram equalization (CLAHE) is the usual next step. Frangi expects **bright tubes**, so we invert the enhanced green channel.

In [ ]:
gray = prepare_gray(image, invert=True)
green = green_channel(image)
fig = make_subplots(rows=1, cols=3, subplot_titles=("RGB fundus", "green channel", "CLAHE + invert"))
fig.add_trace(go.Image(z=image), 1, 1)
fig.add_trace(go.Heatmap(z=green, colorscale="Gray", showscale=False), 1, 2)
fig.add_trace(go.Heatmap(z=gray, colorscale="Gray", showscale=False), 1, 3)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(template="plotly_white", height=360, title=f"Preprocess — {sample.sample_id}")
fig.show()

## Frangi vesselness + Otsu

Frangi's filter measures how the Hessian eigenvalues look like a tube, across several Gaussian scales. Thin capillaries need small σ, the optic-disc trunks need larger σ. We take the maximum response, mask to the FOV, and threshold with Otsu.

In [ ]:
pred, resp = segment_frangi(image, fov)
m = segmentation_metrics(pred, vessel, fov)
print({k: round(m[k], 3) for k in ("dice", "iou", "sensitivity", "specificity", "precision")})

fig = make_subplots(rows=1, cols=3, subplot_titles=("vesselness", "Frangi mask", "ground truth"))
fig.add_trace(go.Heatmap(z=resp, colorscale="Viridis", showscale=False), 1, 1)
fig.add_trace(go.Heatmap(z=pred.astype(float), colorscale="Gray", showscale=False), 1, 2)
fig.add_trace(go.Heatmap(z=vessel.astype(float), colorscale="Gray", showscale=False), 1, 3)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(template="plotly_white", height=360, title="Classical segmentation")
fig.show()

## Test-set metrics

Dice and IoU summarise overlap. **Sensitivity** is the one that hurts when capillaries disappear; **precision** is the one that hurts when the choroid is mistaken for a vessel. All scores ignore pixels outside the FOV (DRIVE convention).

In [ ]:
rows = []
for s in iter_samples(root, split="test"):
    img, gt, f = load_sample(s)
    p, _ = segment_frangi(img, f)
    r = segmentation_metrics(p, gt, f)
    r["sample_id"] = s.sample_id
    rows.append(r)
df = pd.DataFrame(rows)
print(df[["sample_id", "dice", "iou", "sensitivity", "specificity", "precision"]].round(3))

long = df.melt(value_vars=["dice", "iou", "sensitivity", "specificity", "precision"],
               var_name="metric", value_name="value")
fig = px.box(long, x="metric", y="value", points="all", range_y=[0, 1],
             title="Frangi metrics on the demo test split")
fig.update_layout(template="plotly_white")
fig.show()